# EVA Colab Training

**Model**: D=2560, 24 layers, 32 experts, SwiGLU, QK-RMSNorm, RoPE freq=1e6, pos_id binding
**Run**: seq=256 fp32 → ~15.3 GB, ~70 tok/s on L4 22.5GB; budget horizon 150k steps (≈38M tokens of the 1.93B corpus)
**Data**: token_stream_*.bin files in Google Drive; last 3 files = file-level hold-out (audit M13)

---

In [ ]:
# @title 1. Mount Drive & Install Deps
import os, sys, math, time, glob, json, gc

# CUDA allocator: reduce fragmentation on T4 (24 layers x D=2560)
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

from google.colab import drive
drive.mount('/content/drive')

# Config -- point this to your Drive folder with eva_clm/ and data/
DRIVE_ROOT = '/content/drive/MyDrive/eva_clm'  # @param {type:'string'}
DATA_DIR    = os.path.join(DRIVE_ROOT, 'data')
SAVE_DIR    = os.path.join(DRIVE_ROOT, 'checkpoints')
LOG_DIR     = os.path.join(DRIVE_ROOT, 'logs')

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print(f'DRIVE_ROOT={DRIVE_ROOT}')
print(f'DATA_DIR={DATA_DIR}')
print(f'SAVE_DIR={SAVE_DIR}')


In [ ]:
# @title 2. Clone Code from GitHub (data on Drive)
import subprocess

DST = '/content/eva_clm'
if not os.path.exists(DST):
    print('Cloning from GitHub...')
    subprocess.run(['git', 'clone', 'https://github.com/BlackCatSpb/EVA-CLM.git', DST], check=True)
    print('Done.')
else:
    print('Already cloned, force-pulling latest...')
    subprocess.run(['git', '-C', DST, 'fetch', 'origin', 'master'], check=True)
    subprocess.run(['git', '-C', DST, 'reset', '--hard', 'origin/master'], check=True)

sys.path.insert(0, DST)
os.chdir(DST)
print(f'Working dir: {os.getcwd()}')


In [ ]:
# @title 3. Verify GPU & Imports
import torch
import torch.nn.functional as F
import numpy as np
import math
from torch.serialization import add_safe_globals
from core import EVAConfig, EVAStack, MirrorLRScheduler

add_safe_globals([EVAConfig])

device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu_name = torch.cuda.get_device_name(0) if device == 'cuda' else 'N/A'
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9 if device == 'cuda' else 0
print(f'Device: {device}  GPU: {gpu_name}  VRAM: {gpu_mem:.1f} GB')

# A100 (Ampere+) only: fp32 matmuls go through TF32 tensor cores (~10-bit
# mantissa, per-op rel.err ~1e-3 — orders below gradient noise and the eval
# spread). On T4/V100 these flags are no-ops for TC-less GPUs (T4 has no
# TF32; the setting is ignored there), so this is safe to leave on.
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print(f'TF32: ON   bf16 supported: {torch.cuda.is_bf16_supported() if device == "cuda" else False}')
print(f'PyTorch: {torch.__version__}  CUDA: {torch.version.cuda}')
# M16: enable the expandable-segments allocator for real. Colab's kernel
# imports torch before cell 1 can set the env var, so PYTORCH_CUDA_ALLOC_CONF
# never applied (both live OOM messages kept recommending it). Private-API
# setter; guarded -> older torch / CPU runtimes are a no-op.
try:
    torch.cuda.memory._set_allocator_settings('expandable_segments:True')
    print('cuda allocator: expandable_segments=ON')
except Exception as _ae:
    print(f'cuda allocator: expandable_segments unavailable ({type(_ae).__name__})')


In [ ]:
# @title 4. Build Model (L4: D=2560, 24 layers, seq_len=256)
gc.collect()
torch.cuda.empty_cache()

cfg = EVAConfig(
    D=2560,
    n_layers=24,
    bind_K=32,
    code_dim=64,  # B2: twin_free codebook needs K=64 for 65 536 words at overlap≤S−2 (d=D/K=40)
    codebook='twin_free',  # B2: no overlap-5 twins — measured recall knee T 550→≥1200
    vocab=65536,
    mask_eos=False,  # do not mask EOS (separate files *_eos.bin)
    mlp_groups=32,
    mlp_expand=4,
    seq_len=512,  # A100 40GB: cache-attention is O(L^2*entries) — paired with max_entries=12 + ckpt (below)
    lr=3e-4,
    max_steps=150_000,  # ~38M tokens = 2% of the 1.93B corpus in ONE Colab-Pro budget cycle (~160-190h of ~307h left);
                                # the run is resumable — cursor/RNG/val-history all ride in best.pt (M12)
    warmup_steps=1200,  # NOTE: __post_init__ пересчитывает warmup/log/eval из lambda-дома — см. пост-init ниже
    log_interval=55,
    eval_interval=1045,  # NOTE: тоже перетирается __post_init__ — см. ниже
    optimizer='eva_proj',  # A2 arm: EVAAdamW + AdamP-projection. A0='adamw', A1='eva'
    save_interval=1000,
    scheduler='mirror',
    lr_boost_max=2.0,  # upward LR path: allow LR to climb above base when val on downtrend (0=disable)
    lr_improve_tol=0.002,  # downtrend tolerance for the boost gate (hysteresis vs eval noise)
    per_layer_ls_lr=True,  # per-layer LR modulation from fast/slow EMA var(log_scale)
    ls_ema_fast=0.99,
    ls_ema_slow=0.999,
    ls_mult_min=0.5,
    ls_mult_max=2.0,
    ls_mirror_mult_max=2.0,
    private_mem=True,
    expert_asymmetry=True,
    meta_trust=True,
    data_dir=DATA_DIR,
    save_dir=SAVE_DIR,
    log_dir=LOG_DIR,
    conv_kernel=48,
    gradient_checkpointing=True,  # A100: recompute is cheap vs VRAM; keeps 85% governor threshold idle
    head_mode='sigmoid_coded',
    embed_center=False,  # B9 (audit 02a): kills codebook common-mode (pair-cos 0.951->0.000, roundtrip 1.000). GEOMETRY CHANGE — set True on a FRESH run only.
    head_normalize=True,
    bind_twist_mode='trajectory_spiral',
    bind_traj_dims=3,
    hybrid_alpha_max=0.7,
    hybrid_alpha_min=0.3,
    w_pred_scale_init=3.0,
    bind_twist_gate=True,
    collective_layer=True,
    collective_layer_idx=None,
    collective_read_out=True,
    collective_uncert_theta=0.5,
    collective_uncert_kappa=3.0,
    collective_contra_thresh=-0.1,
    collective_contra_gain=6.0,
    collective_maturity_thresh=0.12,
    surprisal_weight=0.3,
    branch_balance_weight=0.1,
    variable_precision=True,
    precision_threshold=0.3,
    explicit_reasoning=True,
    reasoning_max_steps=8,
    reasoning_adaptive=True,  # adaptive-depth reasoning loop (gated by knowledge signals)
    use_amp=False,  # cycle-1 fp32 clean baseline; bf16 crash fixed in B10b (UCL dtype) — flip True after the first EVAL for ~2x on A100
    intent_bridge=True,  # Intent Bridge: top-down/bottom-up intent signal (unbounded context)
    bridge_glu=True,  # BridgeGLU: live semantic MLP gate (replaces frozen mod_scale_mlp; no boost hack needed)
    bridge_conn=0.1,  # aux: bridge learns connectivity by predicting next token
    maturation_enabled=True,  # unified wake-up gate: live mod / mem-write / bridge-inject / intent gated by layer maturity
    pm_write_delay=0,  # maturity-only write gate (no random-regime echo floor)
    intent_topdown=True,  # top-down intent_state propagation
    memory_bank=True,  # Streaming Memory Bank: L1 buffer + L2 bank + L3 concepts
    mem_l1_slots=3,  # L1: rolling buffer of last 3 sentences (immediate)
    mem_l2_slots=32,  # L2: learned bank with 32 slots (short-term)
    mem_min_write_mat=0.3,  # min maturation before writes allowed (like private_mem)
    mem_bridge_dim=256,  # memory bank bridge dim (matches bridge_dim)
    concept_birth_novelty_threshold=0.15,  # birth only if d_min > threshold
    unified_concept_layer=True,  # unified concept layer (global, after embedding)
    unified_concept_S=8,  # number of concept prototypes
    logit_cache_enabled=True,  # decision #3: code-space cache wired into forward (bias -10 gate => true identity at init)
    logit_cache_max_entries=12,  # A100/L512: window 12x512=6144 keys == same attention load as the old 32x256 (T4 sizing OOMed at 32x384=12288: 33.6GB)
    logit_cache_n_heads=8,  # attention heads for logit cache
    logit_cache_scheduled_sampling=0.05,  # R1: 5% inference-mode during training
    logit_cache_reset_on_resume=True,  # R6: clear cache on resume/LR-reset
    orth_weight=0.0,  # ortho-gram aux off (32x 2560^2 gram graphs are huge)
    div_weight=10.0,  # sigmoid-bounded log_scale divergence
)

# EVAConfig.__post_init__ пересчитывает warmup_steps/log_interval/eval_interval
# из lambda-дома (lambda_utils.LambdaConfig) и ЗАТИРАЕТ аргументы конструктора.
# Поэтому фиксируем явно ПОСЛЕ создания cfg — одинаково для всех рук A/B
# (иначе lambda-дом даёт warmup~101, eval~233, что для 723M слишком резко).
cfg.warmup_steps = 1200
cfg.log_interval = 55
cfg.eval_interval = 1045  # 1045 = 19*55 (выравнивание оценок с логом)
print(f'Overrides: warmup={cfg.warmup_steps}, log_interval={cfg.log_interval}, eval_interval={cfg.eval_interval}')

model = EVAStack(cfg).to(device)
n_params = model.param_count()
print(f'Model: {n_params:,} params ({n_params/1e6:.2f}M)')
print(f'Head: {cfg.head_mode} (normalize={cfg.head_normalize})')
print(f'Bind: {cfg.bind_twist_mode} (gate={cfg.bind_twist_gate})')
print(f'Collective: maturity={cfg.collective_maturity_thresh}, read_out={cfg.collective_read_out}')
print(f'Variable Precision: {cfg.variable_precision} (threshold={cfg.precision_threshold})')
print(f'Explicit Reasoning: {cfg.explicit_reasoning} (max_steps={cfg.reasoning_max_steps}, adaptive={cfg.reasoning_adaptive})')
print(f'AMP: {cfg.use_amp}')

gc.collect()
torch.cuda.empty_cache()


In [ ]:
# @title 5. Mixed Precision Setup (B=1 fits in T4)
cfg.batch_size = 1
print(f'Batch size: {cfg.batch_size}')
print(f'  Tokens/step: {cfg.batch_size * cfg.seq_len}')

# AMP policy (A100): cfg.use_amp=True in cell 4 enables bf16 autocast around
# the FORWARD (tensor cores + big activation-RAM saving — the real reason the
# seq-512 run sits at 74% VRAM). fp16 deliberately NOT used: the collective
# layer overflows it (that's why T4 ran fp32). bf16 needs no GradScaler:
# exponent range == fp32. Loss ops (CE/BCE) are on autocast's fp32 list and
# are promoted automatically, so the coded-head NLL keeps full precision.
_USE_AMP = bool(getattr(cfg, 'use_amp', False)) and device == 'cuda'
_AMP_DTYPE = torch.bfloat16 if (_USE_AMP and torch.cuda.is_bf16_supported()) else torch.float16
_USE_AMP = _USE_AMP and _AMP_DTYPE is torch.bfloat16  # fp16 fallback disabled by design
scaler = None  # bf16: no loss scaling needed


In [ ]:
# @title 6. Optimizer mode (A/B arms)
# NOTE: the real optimizer is built in cell 8b via core.adaptation.build_optimizer
# with optimizer=getattr(cfg, 'optimizer', 'adamw'). This cell only validates the mode.

import torch
from core.adaptation import build_optimizer

_mode = getattr(cfg, 'optimizer', 'adamw')
assert _mode in ('adamw', 'eva', 'eva_proj'), f'unknown optimizer={_mode!r}'
_probe = build_optimizer(model, cfg.lr, llrd_decay=1.0,
                         weight_decay=cfg.weight_decay, betas=(0.9, 0.95),
                         optimizer=_mode)
if _mode in ('eva', 'eva_proj'):
    print(f'Optimizer: EVAAdamW mode={_probe.defaults["mode"]} '
          f'projected={_probe.defaults["projected"]} '
          f'groups={len(_probe.param_groups)}')
else:
    print(f'Optimizer: torch.optim.AdamW groups={len(_probe.param_groups)}')
del _probe, _mode
print(f'cfg.optimizer = {cfg.optimizer} (A0=adamw | A1=eva | A2=eva_proj)')


In [ ]:
# @title 7. Data Streams
class TokenStream:
    def __init__(self, path):
        self.data = np.memmap(path, dtype=np.uint16, mode='r')
        self.path = path
        self.len = len(self.data)
    def get_batch(self, seq_len, batch_size, offset, vocab=None):
        needed = batch_size * seq_len + 1
        wrapped = offset + needed > self.len
        if wrapped:
            offset = 0
        chunk = self.data[offset:offset + needed]
        # B7 (audit 01 F-01): loud corpus/vocab mismatch, no silent clip.
        if vocab is not None:
            _hi = int(chunk.max(initial=0))
            if _hi >= vocab:
                raise ValueError(f'TokenStream: id {_hi} >= vocab {vocab} in {self.path} - corpus/model mismatch')
        x = torch.from_numpy(chunk[:batch_size * seq_len].reshape(batch_size, seq_len).copy())
        y = torch.from_numpy(chunk[1:batch_size * seq_len + 1].reshape(batch_size, seq_len).copy())
        # Audit M8: explicit `wrapped` flag — the old 3-tuple let get_batch
        # silently rewind the offset, so the caller's rotation branch (new
        # stream + document-state reset) only ever ran at step 0 and one
        # stream served the whole run.
        return x.long(), y.long(), offset + batch_size * seq_len, wrapped

stream_files = sorted(glob.glob(os.path.join(DATA_DIR, 'token_stream_*_clean.bin')))
if not stream_files:
    stream_files = sorted(glob.glob(os.path.join(DATA_DIR, 'token_stream_*.bin')))
if not stream_files:
    print('WARNING: No token_stream_*.bin found!')
    print(f'  Looked in: {DATA_DIR}')
    print('  Using random data for testing')
    streams = []
else:
    streams = [TokenStream(f) for f in stream_files]
    total_tokens = sum(s.len for s in streams)
    print(f'Found {len(streams)} files, {total_tokens:,} total tokens')
    # Audit M13: the M8 hold-out was ONE file — val_loss was a single
    # domain-draw of the corpus. With 39 files, hold out the last three
    # (file-level, never trained on) and average val over them; per-file
    # batch budget is divided so total eval cost stays ~constant. Small
    # corpora (< 8 files) keep the M8 single-stream semantics.
    _hold_n = 3 if len(streams) >= 8 else (1 if streams else 0)
    if streams:
        _hold_files = [os.path.basename(f) for f in stream_files[-_hold_n:]]
        print(f'  train pool: {len(streams) - _hold_n} files | hold-out: {_hold_n} files: '
              + ', '.join(n[-28:] for n in _hold_files))


In [ ]:
# @title 8. Checkpoint (fresh start for A/B)
# B15 (audit 05 F5-02): verify_identity_resume/codebook_fingerprint are used
# HERE but first imported in cell 9 — fresh runtime + FORCE_FRESH=False died
# with NameError. Import at point of need (idempotent).
from core.training_control import codebook_fingerprint, verify_identity_resume
def _t15(o):
    # B15 (F5-07): move restored streaming state (nested tensors) to device
    if torch.is_tensor(o):
        return o.to(device)
    if isinstance(o, (list, tuple)):
        return [_t15(x) for x in o]
    return o
FORCE_FRESH = True  # A/B arms start from scratch (len 256). Set False to RESUME best.pt.
start_step = 0
# B16: FORCE_FRESH must not inherit stale resume vars from a prior run
_resume_cuda_rng = None
resumed_active_depth = None  # set on resume; None for fresh start / pre-fix ckpt
state = None
best_val_loss = float('inf')
# M12 single-best.pt: everything a restart needs rides in the checkpoint;
# these hold the restored values (defaults = fresh-start cursor).
resumed_stream_idx = 0
resumed_offset = 0
_resume_detector_sd = None
_resume_balancer_sd = None
_resume_rng = None
_resume_data_rng = None

best_ckpt = os.path.join(SAVE_DIR, 'best.pt')

# Find the BEST numbered checkpoint (best N.pt) by step number.
# best.pt may be overwritten with step=0 data; numbered files are immutable.
def _find_best_checkpoint(save_dir):
    import re
    numbered = []
    for f in glob.glob(os.path.join(save_dir, 'best [0-9]*.pt')):
        m = re.search(r'best\s+(\d+)\.pt$', os.path.basename(f))
        if m:
            try:
                ckpt = torch.load(f, map_location='cpu', weights_only=False)
                step = ckpt.get('step', 0)
                val = ckpt.get('best_val_loss', float('inf'))
                numbered.append((f, step, val))
            except Exception:
                pass
    if numbered:
        # Prefer highest step, then lowest val_loss
        numbered.sort(key=lambda x: (-x[1], x[2]))
        return numbered[0]
    return None

# Single-checkpoint policy: prefer best.pt if it has real progress;
# fall back to numbered checkpoints if best.pt is stale (step=0).
ckpt_files = []
if os.path.exists(best_ckpt) and not FORCE_FRESH:
    try:
        _ckpt = torch.load(best_ckpt, map_location='cpu', weights_only=False)
    except RuntimeError as e:
        print(f'  best.pt CORRUPTED ({e}), falling back to numbered checkpoints...')
        _ckpt = {'step': 0}
    if _ckpt.get('step', 0) > 0:
        ckpt_files = [best_ckpt]
    else:
        if not ckpt_files:
            print(f'  best.pt has step=0 (stale or corrupted), looking for numbered checkpoints...')
        result = _find_best_checkpoint(SAVE_DIR)
        if result:
            best_ckpt, _step, _val = result
            ckpt_files = [best_ckpt]
            print(f'  Found best numbered checkpoint: {os.path.basename(best_ckpt)} (step={_step}, val={_val:.4f})')
            import shutil
            shutil.copy2(best_ckpt, os.path.join(SAVE_DIR, 'best.pt'))
            print(f'  Copied to best.pt')
    del _ckpt

if (not FORCE_FRESH) and ckpt_files:
    # Prefer FULL (uncompressed) checkpoints so optimizer/scheduler survive resume.
    # FCF-CPR compressed checkpoints STRIP optimizer state -> fresh Adam -> instability.
    def _is_full(p):
        s = os.path.basename(p)[:-3]  # drop '.pt'
        return ('_fcf' not in s) and ('_cpr' not in s)
    full = [p for p in ckpt_files if _is_full(p)]
    if full:
        latest = full[-1]
        if latest != ckpt_files[-1]:
            print(f'  Preferring full checkpoint {latest} over newer compressed one (keeps optimizer state)')
    else:
        latest = ckpt_files[-1]
        if any(('_fcf' in os.path.basename(p)) or ('_cpr' in os.path.basename(p)) for p in ckpt_files):
            print('  WARNING: resumed from COMPRESSED checkpoint - optimizer/scheduler state absent -> fresh Adam, unstable. Use a full step_*.pt for clean resume.')
    print(f'Resuming from {latest}')
    ckpt = torch.load(latest, map_location=device, weights_only=False)
    resumed_active_depth = ckpt.get('active_depth', None)  # restore true depth if present
    from core.migrate import migrate_state_dict
    sd, n_mig = migrate_state_dict(dict(ckpt['model']), model)
    if n_mig:
        print(f'  MIGRATED {n_mig} keys (W_out +K, bind_coh_gate=0, freq_scale=1.0)')
    # Filter size-mismatched keys (e.g. L2 slots changed 16->32)
    _model_sd = model.state_dict()
    _skipped = []  # B8: identity drift is fatal
    _filtered = {}
    for k, v in sd.items():
        if k in _model_sd and _model_sd[k].shape != v.shape:
            print(f'  SKIP size-mismatch: {k} ckpt={list(v.shape)} model={list(_model_sd[k].shape)}')
            _skipped.append(k)
        else:
            _filtered[k] = v
    miss, unex = model.load_state_dict(_filtered, strict=False)
    verify_identity_resume(model, ckpt, _skipped)  # B8 (02a F2A-03)
    _resume_cuda_rng = ckpt.get('cuda_rng')   # B16 (applied in cell 8 directly)
    # B15 (audit 05 F5-07): mid-document resume continues the DOCUMENT —
    # per-layer VSA state + global_state used to live outside best.pt, so every
    # Colab restart cold-started the current document (measured 0.37 nat).
    # B16 (audit 05 follow-up): the captured state is a LIST holding
    # tensors — truthiness must go through len(), bool(list_of_tensors)
    # evaluates the tensor ('Boolean value of Tensor ... ambiguous').
    if ckpt.get('stream_state') is not None and len(ckpt['stream_state']):
        state = _t15(ckpt['stream_state'])
    if ckpt.get('stream_gs') is not None and len(ckpt['stream_gs']):
        gs = _t15(ckpt['stream_gs'])

    # fix: reopen cognitive gate on resume — hybrid_gate (sigmoid+softmax)
    # replaces frozen mod_scale_mlp. Initialize tau for per-expert specialization.
    # B15 (F5-09): reopen only for checkpoints that PREDATE hybrid_gate —
    # it fired every resume and clobbered the RESTORED learned gate bias and
    # log_tau with config defaults.
    if getattr(cfg, 'mlp_gate_b_init', 0.0) > 0 and not any(k.endswith('mlp_gate_b') for k in _filtered):
        tau_val = getattr(cfg, 'mlp_hybrid_gate_tau', 1.0)
        for layer in model.layers:
            layer.mlp.mlp_gate_b.data.fill_(cfg.mlp_gate_b_init)
            layer.mirror.hybrid_gate.log_tau.data.fill_(math.log(tau_val))
        print(f'  reopened cognitive gate: mlp_gate_b -> {cfg.mlp_gate_b_init}, hybrid_gate tau -> {tau_val:.3f}')
        if getattr(cfg, 'bridge_glu', False):
            print('  bridge_glu=True: BridgeGLU params (bridge_glu_net.*) absent in old ckpts -> start fresh, will adapt.')

    # fix: private-mem write step not saved in pre-fix checkpoints.
    # _pm_step is a conditional buffer (exists only when cfg.private_mem is on),
    # so guard the attribute — private_mem=False models never had it and must
    # not crash on resume (this path only fires when FORCE_FRESH=False).
    pm_key = 'layers.0.mirror._pm_step'
    if pm_key not in ckpt['model'] and hasattr(model.layers[0].mirror, '_pm_step'):
        for l in model.layers:
            if hasattr(l.mirror, '_pm_step'):
                l.mirror._pm_step.fill_(5000)
        print('  _pm_step not saved (pre-fix checkpoint): private-mem write enabled now')

    if miss:
        print(f'  Missing keys: {len(miss)}')
    if unex:
        print(f'  Unexpected keys: {len(unex)}')

    def _restore_optimizer(optimizer, model, ckpt):
        ckpt_opt = ckpt.get('optimizer', {}) or {}
        old_names = ckpt.get('param_names')
        if old_names is None:
            old_names = ckpt_opt.get('param_names')  # legacy location
        if old_names is None:
            print('  WARNING: no param_names - optimizer state NOT restored (fresh Adam)')
            return False
        names = {id(p): n for n, p in model.named_parameters()}
        pos = {id(p): i for i, p in enumerate(
            (p for g in optimizer.param_groups for p in g['params']))}
        new_sd = optimizer.state_dict()
        new_sd['state'] = {}
        old_state = ckpt_opt.get('state', {})
        old_groups = ckpt_opt.get('param_groups', [])
        moved = skipped = 0
        for name, p in model.named_parameters():
            if name not in old_names:
                skipped += 1
                continue
            si = old_names.index(name)
            st = old_state.get(str(si)) if str(si) in old_state else old_state.get(si)
            if st is None:
                continue
            if tuple(st['exp_avg'].shape) != tuple(p.shape):
                if (name.endswith('bind.W_out') and len(st['exp_avg'].shape) == 2
                        and st['exp_avg'].shape[1] == p.shape[1]
                        and st['exp_avg'].shape[0] < p.shape[0]):
                    st = {k: (v[:p.shape[0]] if isinstance(v, torch.Tensor) and v.dim() == 2 else v)
                          for k, v in st.items()}
                    moved += 1
                else:
                    skipped += 1
                    continue
            else:
                moved += 1
            new_sd['state'][pos[id(p)]] = {k: (v.clone() if isinstance(v, torch.Tensor) else v)
                                            for k, v in st.items()}
        for gi in range(min(len(new_sd['param_groups']), len(old_groups))):
            if 'lr' in old_groups[gi]:
                new_sd['param_groups'][gi]['lr'] = old_groups[gi]['lr']
        optimizer.load_state_dict(new_sd)
        print(f'  Optimizer restored by name: {moved} slots, {skipped} skipped')
        return moved > 0
    start_step = ckpt.get('step', 0)
    best_val_loss = ckpt.get('best_val_loss', float('inf'))
    reasoning_enabled_step = ckpt.get('reasoning_enabled_step', 0)
    # M12: watchdog/balancer stats, data cursor and RNG streams (old ckpts
    # without these keys resume with defaults — recover_count pre-M12 was
    # saved as a top-level key; honour it inside the detector state).
    _resume_detector_sd = ckpt.get('detector')
    _resume_depth_state = ckpt.get('depth_state')  # B14 (F4-06)
    if _resume_detector_sd is None and 'recover_count' in ckpt:
        _resume_detector_sd = {'recover_count': int(ckpt.get('recover_count', 0) or 0)}
    _resume_balancer_sd = ckpt.get('balancer')
    _resume_rng = ckpt.get('rng')
    _resume_data_rng = ckpt.get('data_rng')
    resumed_stream_idx = int(ckpt.get('stream_idx', 0) or 0)
    resumed_offset = int(ckpt.get('offset', 0) or 0)
    print(f'  Resumed at step {start_step}, reasoning ramp t={reasoning_enabled_step}')
else:
    reasoning_enabled_step = 0
    print('No checkpoint found, starting fresh')


In [ ]:
# @title 8b. Adaptation module (unified, principled -- core.adaptation)
# Single source of truth for training stability. Replaces the old scattered
# guard: progressive depth (validation-plateau driven, not a fixed schedule),
# LR (linear warmup + mirror-adaptive multiplier + plateau damping + rewind on
# recovery), statistical failure detection (3 sigma CE rule), adaptive gradient
# clipping (AGC), and aux-loss balancing via spectral gradient alignment
# (no per-loss magic weights).
from core.adaptation import (LossBalancer, DepthController, LRController,
                             FailureDetector, GradientClipper, hard_veto_ceiling,
                             codebook_fingerprint, verify_identity_resume,
                             apply_tau_lr,
                             set_active_depth, build_optimizer)

# Guard: cell '8. Resume Checkpoint' MUST run before this one. It loads best.pt
# and defines start_step / resumed_active_depth / best_val_loss / ckpt_files /
# reasoning_enabled_step. Running this cell alone (e.g. after a kernel restart)
# leaves them undefined -> the cryptic "NameError: start_step" and, worse, a
# silent fresh start from step 0 (which we never want; we always resume best.pt).
for _need in ('start_step', 'resumed_active_depth', 'best_val_loss', 'ckpt_files',
              'reasoning_enabled_step', 'FORCE_FRESH', 'best_ckpt',
              'resumed_stream_idx', 'resumed_offset', '_resume_detector_sd',
              '_resume_balancer_sd', '_resume_rng', '_resume_data_rng'):
    if _need not in globals():
        raise RuntimeError(
            "Cell '8. Resume Checkpoint' was not executed. Run it BEFORE this cell "
            "(or Runtime -> Restart session -> Run all). It loads best.pt and defines the needed state.")


def _make_opt(lr):
    cfg.lr = float(lr)
    # llrd_decay=1.0: disable index-based LLRD (replaced by tau_config.lr_mult in training loop)
    # optimizer mode flows from cfg.optimizer ('adamw' | 'eva' | 'eva_proj')
    return build_optimizer(model, lr, llrd_decay=1.0,
                           weight_decay=cfg.weight_decay, betas=(0.9, 0.95),
                           optimizer=getattr(cfg, 'optimizer', 'adamw'))

# 3) Progressive unfreeze driven by validation-loss plateau (diminishing
#    returns), NOT a fixed schedule. On resume continue from the checkpoint
#    depth so all-24 never wake at once on a memory-tight T4.
starting_fresh = FORCE_FRESH or not ckpt_files

# 1) Optimizer — restore from checkpoint if available, else fresh Adam.
optimizer = _make_opt(cfg.lr)
if not starting_fresh and '_restore_optimizer' in dir():
    # Use name-based restore (handles architecture changes between saves)
    if not _restore_optimizer(optimizer, model, ckpt):
        print('  WARNING: optimizer state not restored (fresh Adam)')
elif not starting_fresh and 'optimizer' in ckpt and ckpt.get('optimizer') is not None:
    try:
        optimizer.load_state_dict(ckpt['optimizer'])
        print('Optimizer restored from checkpoint (momentum preserved)')
    except Exception as e:
        print(f'[warn] optimizer restore failed: {e} — using fresh optimizer')
else:
    print('Optimizer: fresh (no checkpoint state)')

# 2) LR controller: warmup + mirror-adaptive multiplier + plateau damping.
#    Restore full scheduler state (EMA baselines, val_ema, loss_lr_factor)
#    to avoid LR instability after resume.
scheduler = LRController(model, optimizer, cfg=cfg)
if not starting_fresh and 'scheduler' in ckpt and ckpt.get('scheduler') is not None:
    scheduler.load_state_dict(ckpt['scheduler'])
    print(f'Scheduler state restored (step={start_step})')
else:
    scheduler.set_step(start_step)
    print(f'Scheduler: fresh (step={start_step})')

if starting_fresh:
    init_k = 8
else:
    # Restore the TRUE achieved active depth from the checkpoint (not the
    # step-based heuristic) so frequent stop/resume doesn't keep resetting to
    # 8 active layers (which would freeze deep layers forever). Falls back to
    # the legacy heuristic for pre-fix checkpoints that lack 'active_depth'.
    init_k = resumed_active_depth if resumed_active_depth is not None \
        else min(8 + (start_step // 15000) * 4, cfg.n_layers)
depth = DepthController(model, n_layers=cfg.n_layers, init_k=init_k,
                        unfreeze_inc=4, eval_interval=cfg.eval_interval)
set_active_depth(model, init_k)
print(f'Progressive unfreeze: init_active={init_k} (fresh={starting_fresh}, start_step={start_step})')

# 4) Aux-loss balancer: spectral alignment bounds the aux gradient by ||g_CE||.
balancer = LossBalancer(align=True, align_cap=10.0, eval_interval=cfg.eval_interval)

# 5) Relative failure detector (value > slow_ema·(1+rel_margin), multi-signal)
# Decision D6 (post-M15): the FailureDetector is a PURE ALARM SENSOR. Sustained
# relative divergence stops the loop; there is NO automatic rollback — the
# emergency actuator was twice the cause of fatal incidents (fragmented CUDA
# allocator + 3.4GB torch.load at divergence time) and never cured a root
# cause. Continuous self-regulation stays in the adaptive layer (MirrorLR,
# AGC, maturation, M14 soft veto); discrete recovery is a human call made from
# a clean best.pt (last val-improving save) and the log forensics.
watchdog = FailureDetector(model, k_sigma=3.0, warmup=cfg.warmup_steps)  # B4: margins/floors are class defaults now
# M12: watchdog statistics ride in best.pt — signal baselines, violation
# streaks and recover_count survive restarts (recover_count was SAVED by M8
# but never restored until now). The balancer's balance-mode EMAs are also
# persisted (they stay None in the align-mode production config — harmless).
watchdog.load_state_dict(_resume_detector_sd)
# B14 (F4-06): resume the plateau integrator (depth exists by now, cell-9 order)
depth.put_state(_resume_depth_state if '_resume_depth_state' in dir() else None)
# A resumed session is a regime change (fresh CUDA allocator, possibly
# fresh LR path): CE baseline from the old session must re-bootstrap.
watchdog._stats.pop('ce', None)
balancer.load_state_dict(_resume_balancer_sd)
if _resume_detector_sd:
    print(f'  watchdog restored: recover_count={watchdog.recover_count}, '
          f'ce_armed={watchdog.ce_armed}, {len(watchdog._stats)} baselines')
print('FailureDetector: ALARM sensor (D6+ — warns once, loop stops on confirmed second; never self-heals; CE joins after first val eval)')

# 6) Adaptive gradient clipping (AGC): clip iff ||g|| > c*||theta|| (ratio).
# EVA-блоки трансформероподобны -> c=0.1 (docstring: 0.01 — режим ResNet).
clipper = GradientClipper(c=0.1)
# τ-aware AGC: c_eff = c·(τ_ref/τ_l)^γ по τ-лестнице модели (per-layer map).
clipper.attach(model)
orig_seq_len = cfg.seq_len  # capture so recovery can undo OOM-driven seq_len shrink


In [ ]:
# @title 9. TRAINING LOOP (principled adaptation: spectral-aligned aux, plateau depth, 3-sigma watchdog, AGC)
import torch._dynamo
def _d15(o):
    # B15 (F5-07): detach nested streaming state to CPU for checkpoint saving
    if torch.is_tensor(o):
        return o.detach().cpu()
    if isinstance(o, (list, tuple)):
        return [_d15(x) for x in o]
    return o
torch._dynamo.config.suppress_errors = True
import time, os, math  # defensive: ensure available even if this cell runs standalone

stream_idx = resumed_stream_idx   # continue the data cursor (audit M12)
gs = None  # cross-layer stream state, threaded per step (audit M8)
offset = resumed_offset
if streams and stream_idx >= max(len(streams) - _hold_n, 1):
    # cursor saved under the pre-M13 1-file split could point INTO the new
    # 3-file hold-out — rotate to a fresh pool document instead
    stream_idx, offset = 0, 0
intent_state = None
tokens_seen = 0
t0 = time.time()
rng = torch.Generator(device='cpu').manual_seed(42)
if _resume_rng is not None:      # document-shuffle & dropout streams
    torch.set_rng_state(_resume_rng.cpu())   # continue where the ckpt left off (M12; .cpu() -- cell 8 loads with map_location=device, and RNG states MUST be CPU bytetensors)
if _resume_data_rng is not None:
    rng.set_state(_resume_data_rng.cpu())  # CPU-only contract (see above)
_last_oom_step = -10**9  # M15: OOM-regrow cooldown (probe at most once /200 steps)
_alarm_stop = False  # D6: set when the alarm sensor stops the run
_alarm_strikes = getattr(watchdog, '_strikes', 0)   # B16: strikes ride best.pt

# gradalign: gradient-reactive governance loss for the MLP gate (0 = disabled).
cfg.gradalign_weight = 0.3

# Semantic Bridge is now IN-CORE (core/bridge.py): a per-layer SemanticBridge runs
# inside the model forward (train + inference). At every layer it emits a semantic
# vector, predicts the NEXT token's embedding (1 - cos aux loss, computed inside
# model.compute_losses -> aux_dict['bridge_conn']), and injects a persistent
# cross-layer stream back into the hidden state. Enabled via cfg.bridge_conn > 0
# (set in cell 4) at MODEL INIT time. No external head / extra param group => no
# StopIteration at checkpoint save (all bridge params live inside the model).
if getattr(cfg, 'bridge_conn', 0.0) > 0 and getattr(model, 'bridge', None) is not None:
    print(f'In-core SemanticBridge active (bridge_conn={cfg.bridge_conn}, '
          f'bridge_dim={model.bridge.bridge_dim}, params={sum(p.numel() for p in model.bridge.parameters()):,})')
elif getattr(cfg, 'bridge_conn', 0.0) > 0:
    print('WARNING: cfg.bridge_conn>0 but model.bridge is None -> set bridge_conn BEFORE model init.')
bridge_head = None


batch_size = getattr(cfg, 'batch_size', 1)

print(f'Training: step {start_step} -> {cfg.max_steps}')
print(f'  ({cfg.max_steps - start_step} steps remaining)')
print(f'  batch={batch_size} seq={cfg.seq_len} -> tokens/step={batch_size * cfg.seq_len}')

try:
    for step in range(start_step, cfg.max_steps):
        model.train()

        if getattr(model, 'explicit_reasoning', False):
            model.reasoning_enabled_step = reasoning_enabled_step

        if streams:
            # Audit M8: rotate at DOCUMENT boundaries BEFORE the read. The old
            # `if offset == 0` never re-fired (get_batch rewound internally and
            # returned a non-zero offset), so one stream served the whole run
            # and banks/bus/reasoning never reset between documents. The LAST
            # stream is hold-out for eval → training pool is streams[:-1].
            _need = batch_size * cfg.seq_len + 1
            if offset == 0 or offset + _need > streams[stream_idx].len:
                stream_idx = torch.randint(0, max(len(streams) - _hold_n, 1), (1,), generator=rng).item()
                offset = 0
                state = None
                intent_state = None
                gs = None
                if getattr(model, 'bridge', None) is not None:
                    model.bridge.bridge_stream.zero_()
                if getattr(model, 'logit_cache', None) is not None:
                    model.logit_cache.cache.clear()  # new document ⇒ empty cache (decision #3)
                if getattr(model, 'memory_bank', None) is not None:
                    model.memory_bank.reset()
                if getattr(model, 'explicit_reasoning', False):
                    model.reset_reasoning()
            x, y, offset, _wrapped = streams[stream_idx].get_batch(cfg.seq_len, batch_size, offset)
        else:
            x = torch.randint(0, cfg.vocab, (batch_size, cfg.seq_len))
            y = torch.randint(0, cfg.vocab, (batch_size, cfg.seq_len))

        x, y = x.to(device), y.to(device)

        while True:
            try:
              with torch.autocast('cuda', dtype=_AMP_DTYPE, enabled=_USE_AMP):
                h = model.embed_tokens(x)
                out, state, gs, _ = model(h, state, global_state=gs, step=step, intent_state=intent_state, tokens=x)
                model.observe_output(model.lm_head(out))
                ce_loss, aux_dict = model.compute_losses(out, y, h_emb=h)
                # bridge_conn aux loss is produced in-core by model.compute_losses (SemanticBridge).

                # GRADALIGN now produced in-core: core.losses.compute_losses
                # builds aux_dict['gradalign'] from the block backward-hook target
                # (‖∂CE/∂mlp_out‖ per expert) with real cfg.gradalign_weight, and
                # LossBalancer.BYPASS_AUX backprops it directly (audit M5).
                break
            except torch.cuda.OutOfMemoryError:
                # Drop EVERY tensor bound by the failed attempt (and the
                # previous iteration): a retained forward graph (~4GB at
                # seq 256) makes even a seq=64 retry OOM (live incident
                # step 1052: rollback + retained graph = 22GB wall).
                h = out = ce_loss = aux_dict = None
                torch.cuda.empty_cache()
                state = None
                intent_state = None
                gs = None
                if cfg.seq_len > 64 and batch_size == 1:
                    # B19: an OOM at full length means the memory model was
                    # optimistic — turn on recompute FIRST (activations ~3x
                    # smaller, ~30% slower), only halve the window if even
                    # that is not enough (handled by the next retry).
                    if not getattr(cfg, 'gradient_checkpointing', False):
                        cfg.gradient_checkpointing = True
                        print('  [OOM] gradient_checkpointing ON (recompute)', flush=True)
                    else:
                        cfg.seq_len //= 2
                    x, y, offset, _w = streams[stream_idx].get_batch(cfg.seq_len, batch_size, offset)
                    _last_oom_step = step
                    print(f'  [OOM] retry: seq_len={cfg.seq_len} batch={batch_size}')
                    continue
                import gc as _gc
                _gc.collect(); torch.cuda.empty_cache()
                _agg = {}
                for _o in _gc.get_objects():
                    try:
                        if torch.is_tensor(_o) and _o.is_cuda:
                            _k = (tuple(_o.shape), _o.dtype, 'grad' if _o.requires_grad else 'buf')
                            _agg[_k] = _agg.get(_k, 0) + _o.numel() * _o.element_size()
                    except Exception:
                        pass
                    if len(_agg) > 400:
                        break
                _top = sorted(_agg.items(), key=lambda kv: -kv[1])[:10]
                print('  [OOM] top CUDA tensor holders (shape, dtype, kind): bytes')
                for _k, _v in _top:
                    print(f'    {_k}  {_v/1e6:.0f}MB')
                raise

        tokens_seen += batch_size * cfg.seq_len
        intent_state = getattr(model, '_last_intent_state', None)

        ce_val = ce_loss.item()
        # M14 NON-LEARNABLE-BATCH VETO. K*ln2 (32*0.693 = 22.2) is the coded
        # head's uniform-bit NLL — the zero-information ceiling: CE above it
        # means the window is confidently anti-predicting unpredictable ids
        # (corpus garbage stretches; live incident 2026-09: CE drifted 5.8->34
        # ->67 across one bad region while branch/diversity/mirror stayed
        # healthy — a data pathology, not a model one). Its gradient is pure
        # head damage, and letting the watchdog see it loops rollbacks onto
        # the same cursor in best.pt. Veto: no grad, cursor advances, weights
        # and Adam state untouched.
        _ce_uni = hard_veto_ceiling(cfg.vocab)
        if ce_val > _ce_uni:
            print(f'  [veto] step {step}: ce={ce_val:.2f} > {_ce_uni:.1f} (uniform-bit NLL)'
                  ' — non-learnable batch: gradient skipped, cursor advanced', flush=True)
            h = out = ce_loss = aux_dict = None
            optimizer.zero_grad(set_to_none=True)
            continue
        depth.update(step)
        # Multi-signal failure detection: CE + protective metrics (diversity,
        # gate_l1, mlp_out, effective gate amplitude ig_eff). One statistical
        # SPC k·sigma rule per signal (FailureDetector in training_control).
        _mets = {}
        _dv = aux_dict.get('diversity')
        if _dv is not None and isinstance(_dv, torch.Tensor):
            _mets['diversity'] = float(_dv.abs().item())
        _gl = aux_dict.get('gate_l1')
        if _gl is not None and isinstance(_gl, torch.Tensor):
            _mets['gate_l1'] = float(_gl.item())
        with torch.no_grad():
            try:
                # MLP-runaway ratio (self-referencing fast/slow EMA of ‖h_mlp‖):
                # healthy ≈1, ×2–50 runaway lifts it far past the relative margin.
                _rr = [getattr(l, '_mlp_ratio', None) for l in model.layers]
                if _rr and all(v is not None for v in _rr):
                    _mets['mlp_ratio'] = float(max(_rr))
                _ige = [getattr(l.mirror, '_cached_ig_eff', None) for l in model.layers]
                if _ige and all(x is not None for x in _ige):
                    _mets['ig_eff'] = float(sum(_ige) / len(_ige))
            except Exception:
                pass
        # M12 single-checkpoint policy: no rollback.pt writes anymore — the
        # detector restores divergence to best.pt (val-best / step-0 seed).
        # B14 (audit 04 F4-05): LR clock, not loop clock (veto-storm skew).
        _rb = watchdog.check(ce_val, int(getattr(scheduler, '_step', step)), _mets)
        # M14b SOFT veto (margin-tied): the hard K*ln2 ceiling alone does NOT
        # stop LR-escalation divergence (live incident: CE drifted 6.9->34->67
        # across 54 steps through CLEAN, scan-verified data as lr approached
        # the warmup end). A batch >4 rel-margins above the model's own fast
        # EMA level is escalation fuel: training on it feeds the spiral and
        # drags the EMA toward the spike. Skip its gradient, advance cursor.
        if not _rb:
            _fc = watchdog._stats.get('ce')
            if _fc:
                _soft_thr = _fc[0] * (1.0 + 4.0 * watchdog.rel_margin)
                if ce_val > _soft_thr:
                    print(f'  [veto:soft] step {step}: ce={ce_val:.2f} > {_soft_thr:.2f} '
                          f'(fast-EMA escalation gate) — gradient skipped', flush=True)
                    h = out = ce_loss = aux_dict = None
                    optimizer.zero_grad(set_to_none=True)
                    continue
        if _rb:
            # D6+ TWO-STRIKE: the first confirmed sensor signal WARNS and keeps
            # training; a second one after the full cooldown (divergence STILL
            # growing) stops the run. Bounded exposure by design: (a) M14 hard +
            # soft vetoes keep exploding batches out of the weights during the
            # window, (b) best.pt only advances on an improving isolated eval
            # (B3), so continuing cannot poison the saved model, (c) the sensor
            # keeps full authority — only the STOP policy is softened.
            # Rationale (B5 + live 273-incident): in Colab the scarce resource
            # is the SESSION; a false STOP costs hours, the confirmation window
            # costs ~cooldown steps of compute.
            _alarm_strikes = watchdog._strikes   # B16: check() bumped it, persisted
            if device == 'cuda':
                print(f'  [alarm] mem: alloc={torch.cuda.memory_allocated()/1e9:.2f}GB '
                      f'reserved={torch.cuda.memory_reserved()/1e9:.2f}GB '
                      f'alloc_conf={os.environ.get("PYTORCH_CUDA_ALLOC_CONF", "unset")}', flush=True)
            if _alarm_strikes < 2:
                print(f'  [ALARM:WARN] first signal ({_alarm_strikes}/2): continuing training; '
                      'a second confirmed signal after the cooldown will stop the run.', flush=True)
            else:
                _alarm_stop = True
                break

        # Display loss (CE + raw aux sum); gradient uses the balancer.
        with torch.no_grad():
            disp_loss = ce_loss.item() + sum(float(v.item()) if isinstance(v, torch.Tensor) else float(v) for v in aux_dict.values()
                                            if isinstance(v, torch.Tensor))

        # Backward via principled balancer (spectral alignment) + AGC clip.
        if math.isfinite(disp_loss):
                        # B6 (GPT-#34): per-aux gradient geometry vs CE — who drives the model,
            # who fights it, who heats air (printed as name:ratio/cos where ratio
            # = ||g_aux||/||g_CE||). Costs a full backward pass per term, so it runs
            # only at log frequency and never touches .grad or the graph.
            if step and step % cfg.log_interval == 0:
                try:
                    for _m in model.modules():
                        if hasattr(_m, '_step_count') or hasattr(_m, '_alpha_pending'):
                            _m._ggeo_freeze = True
                    try:
                        _gg = balancer.grad_geometry(ce_loss, aux_dict, model.parameters())
                    finally:
                        for _m in model.modules():
                            if hasattr(_m, '_step_count') or hasattr(_m, '_alpha_pending'):
                                _m._ggeo_freeze = False
                    if _gg:
                        print('  [ggeo] ' + '  '.join(
                            f'{k}:{r:.2f}/{c:+.2f}' for k, (r, c) in _gg.items()), flush=True)
                except Exception as _ge:
                    print(f'  [ggeo] unavailable: {_ge}', flush=True)
            # B13 (F4-01): freeze recompute-sensitive side effects around
            # backward (checkpointing re-runs forward inside it).
            for _m in model.modules():
                if hasattr(_m, '_step_count') or hasattr(_m, '_alpha_pending'):
                    _m._ggeo_freeze = True
            try:
                balancer.backward(ce_loss, aux_dict, model.parameters(), phase_model=model)
            finally:
                for _m in model.modules():
                    if hasattr(_m, '_step_count') or hasattr(_m, '_alpha_pending'):
                        _m._ggeo_freeze = False

            # U9 τ-aware AGC: effective clip ratio tightens as the τ-field matures
            # AGC с τ-aware порогом per-layer (c·(τ_ref/τ_l)^γ) — карта
            # построена clipper.attach(model) выше; τ-лестница обучается
            # медленно, перестроим карту раз в eval_interval шагов.
            if step % cfg.eval_interval == 0:
                clipper.attach(model)
            # B13 (audits 03 F3-03 + 04 F4-08, two agents independently):
            # grad-space LR multipliers MUST be applied BEFORE the AGC
            # clip — clipping first and scaling after voids the aux/CE
            # bound on the APPLIED update (measured 7.3-7.7x violations).
            # Per-layer learning rate distribution: scheduler ls_m (mirror
            # log-scale) × tau_config.lr_mult (τ-LLRD). Single source: apply_tau_lr.
            ls_mults = getattr(scheduler, '_ls_mult', None)
            apply_tau_lr(model, getattr(model, 'tau_config', None), ls_mults)
            clipper.clip(model.parameters())
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            if getattr(model, 'explicit_reasoning', False):
                reasoning_enabled_step += 1
            # M15: probe regrow after an OOM-shrink, but at most once per 200
            # steps (a pure-OOM path must not run at 64 forever, nor oscillate)
            if cfg.seq_len < orig_seq_len and step - _last_oom_step > 200:
                _last_oom_step = step
                cfg.seq_len = min(cfg.seq_len * 2, orig_seq_len)
                print(f'  [OOM-recover] probe seq_len -> {cfg.seq_len}', flush=True)
        elif step % cfg.log_interval == 0:
            print(f'  NaN loss at step {step} - skipping step')
            optimizer.zero_grad(set_to_none=True)

        if state is not None:
            state = tuple(
                tuple(t.detach() if isinstance(t, torch.Tensor) else t for t in s)
                if s else None for s in state
            )
        if intent_state is not None:
            intent_state = intent_state.detach()
        gs = gs.detach() if gs is not None else None

        if step % cfg.log_interval == 0:
            dt = time.time() - t0
            tok_s = tokens_seen / max(dt, 1e-8)
            lr = scheduler.get_last_lr()[0]
            mem_gb = torch.cuda.max_memory_allocated() / (1024 ** 3) if device == 'cuda' else 0
            live_gb = torch.cuda.memory_allocated() / (1024 ** 3) if device == 'cuda' else 0  # D6 telemetry: live vs peak
            # M16 memory governor (D6 doctrine: the system holds its own
            # VRAM pulse; humans fix causes, not symptoms). Reserved >85% of
            # capacity -> gradient checkpointing ON (activations recomputed,
            # ~3x smaller footprint, ~30% slower); <70% -> back OFF. The
            # 15% gap is hysteresis so one transient never flaps the switch.
            if device == 'cuda':
                _cap16 = torch.cuda.get_device_properties(0).total_memory
                _v16 = torch.cuda.memory_reserved() / _cap16
                # B19: never decide before a real backward peak exists —
                # at step 0 reserved() reflects forward-only memory, and the
                # first run's premature 'OFF' left no headroom for backward
                # (live OOM at L=512 after the B18 graph additions).
                if step < 60:
                    pass
                elif _v16 > 0.85 and not getattr(cfg, 'gradient_checkpointing', False):
                    cfg.gradient_checkpointing = True
                    print(f'  [memgov] VRAM {_v16:.0%} reserved -> gradient_checkpointing ON', flush=True)
                elif _v16 < 0.70 and getattr(cfg, 'gradient_checkpointing', False):
                    cfg.gradient_checkpointing = False
                    print(f'  [memgov] VRAM {_v16:.0%} reserved -> gradient_checkpointing OFF', flush=True)
            # Effective gate amplitude (normalized in-core by running RMS EMA,
            # blind to raw ‖w_intent‖ growth): ~1.0 healthy, >2 sustained = danger.
            intent_eff = float('nan')
            try:
                _i = [getattr(_l.mirror, '_cached_ig_eff', None) for _l in model.layers]
                if _i and all(x is not None for x in _i):
                    intent_eff = sum(_i) / len(_i)
            except Exception:
                pass
            mod_scl = 0.0
            mod_scl_std = 0.0
            try:
                if any(l.mirror.bridge_glu_net is not None for l in model.layers):
                    _bg = torch.cat([l.mirror._last_mlp_mod.flatten() for l in model.layers])
                    mod_scl, mod_scl_std = _bg.mean().item(), _bg.std().item()
                else:
                    mod_scl = torch.stack([torch.sigmoid(l.mirror.mod_scale_mlp).mean()
                                       for l in model.layers]).mean().item()
            except Exception:
                pass
            # MLP-collapse diagnostics: raw MLP output norm, usefulness gate, maturation gate.
            mlp_out_n = usef_m = mat_g = mat_gmin = mat_gmax = float('nan')
            try:
                with torch.no_grad():
                    if all(getattr(l, '_cache_mlp_out', None) is not None for l in model.layers):
                        mlp_out_n = torch.stack([l._cache_mlp_out.detach().norm() for l in model.layers]).mean().item()
                    usef_m = torch.stack([l.mirror._cached_usefulness.detach().mean() for l in model.layers]).mean().item()
                    if getattr(model, 'maturation', None) is not None:
                        _g = model.maturation.gate.detach()
                        mat_g, mat_gmin, mat_gmax = _g.mean().item(), _g.min().item(), _g.max().item()
            except Exception:
                pass
            # Memory Bank diagnostics
            mb_diag = ''
            if getattr(model, 'memory_bank', None) is not None:
                try:
                    _mb = model.memory_bank.get_diagnostics()
                    mb_diag = f" L1={_mb['l1_write_idx']} L2={_mb['l2_write_idx']}({_mb['l2_consumed']}c) L3={_mb['l3_n_concepts']}({_mb['l3_n_births']}b) scale={_mb['mem_scale']:.3f}"
                except Exception:
                    pass
            # Merge aux_dict + _cached_losses (τ-gating diagnostics live there)
            _lc = getattr(model, '_cached_losses', {})
            _merged = dict(aux_dict)
            for _k, _v in _lc.items():
                if _k not in _merged:
                    _merged[_k] = _v
            aux_str = ' '.join(f'{k}={v:.4f}' for k, v in sorted(_merged.items()) if abs(v.item() if isinstance(v, torch.Tensor) else float(v)) > 1e-6)
            print(f'step={step:>6}  loss={disp_loss:.4f}  ce={ce_loss.item():.4f}  '
                  f'mod_mlp={mod_scl:.3f} mod_std={mod_scl_std:.3f} lr={lr:.2e}  tok/s={tok_s:.0f}  '
                  f'mem={mem_gb:.1f}GB live={live_gb:.1f} d={depth.active}  intent_eff={intent_eff:.4f}  mlp_out={mlp_out_n:.1f} usef={usef_m:.3f}  '
                  f'mat={mat_g:.3f}[{mat_gmin:.3f},{mat_gmax:.3f}]{mb_diag}')
            if aux_str:
                print(f'  aux: {aux_str}')
            if device == 'cuda':
                torch.cuda.reset_peak_memory_stats()

        if step > 0 and step % cfg.eval_interval == 0:
            model.eval()
            _lcache = getattr(model, 'logit_cache', None)
            if _lcache is not None:
                _lcache.cache.clear()  # eval isolation for the cache list (decision #3)
            _rt_snap = model.snapshot_runtime_buffers()  # val must not touch train working memory (M8)
            if getattr(model, 'explicit_reasoning', False):
                model.reset_reasoning()
            # Hold-out eval (audit M13): average over the last _hold_n FILES,
            # each read from its 3/4-region (no wrapped re-read, no train
            # overlap). Per-file batch budget 100//_hold_n keeps total eval
            # cost and the val scale history comparable to the M8 era.
            # Fresh hidden state per batch (state=None) => per-token CE.
            eval_pool = []
            if streams:
                eval_pool = streams[-_hold_n:] if _hold_n else [streams[-1]]
                if not any(getattr(s, 'len', 0) >= batch_size * cfg.seq_len + 1
                           for s in eval_pool):
                    eval_pool = [streams[0]]  # tiny corpus: M8 fallback
            val_loss = 0.0
            n_val = 0
            _val_ok = False
            for eval_stream in eval_pool:
                if getattr(eval_stream, 'len', 0) < batch_size * cfg.seq_len + 1:
                    continue  # file too short for even one batch
                with torch.no_grad():
                    voff = max(eval_stream.len // 4, batch_size * cfg.seq_len + 1)
                    _vmax = max(min(100 // max(_hold_n, 1),
                                    eval_stream.len // (batch_size * cfg.seq_len)), 1)
                    for _ in range(_vmax):
                        vx, vy, voff, _vw = eval_stream.get_batch(cfg.seq_len, batch_size, voff)
                        if _vw:
                            break  # end of hold-out region; no wrapped re-read (audit M8)
                        vx, vy = vx.to(device), vy.to(device)
                        h = model.embed_tokens(vx)
                        out, _, _, _ = model(h, None, adaptive=False, tokens=vx)
                        ce, _ = model.compute_losses(out, vy, h_emb=h)
                        val_loss += ce.item()
                        n_val += 1
                        _val_ok = True
            if _val_ok:
                val_loss /= n_val
                val_ppl = math.exp(val_loss) if val_loss < 20 else float('inf')
                print(f'  EVAL step={step}: val_loss={val_loss:.4f} val_ppl={val_ppl:.2e} (n={n_val})', flush=True)
                depth.update(step, val_loss)
                scheduler.report_val_loss(val_loss)
                watchdog.arm_ce()  # trusted regime + warmup-contaminated CE baseline re-bootstrapped
                # B15 (audit 05 F5-06): restore the PRE-EVAL runtime state BEFORE
                # the save — model.state_dict() used to capture 22 eval-mutated
                # persistent keys (UCL concepts, bridge stream) into the CLEAN artifact.
                model.restore_runtime_buffers(_rt_snap)
                _lc15 = getattr(model, 'logit_cache', None)
                if _lc15 is not None:
                    _lc15.cache.clear()
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    model.flush_control_pending()   # B16 (audit 05)
                    _save_path = os.path.join(SAVE_DIR, 'best.pt')
                    _tmp_path = _save_path + '.tmp'
                    torch.save({
                        'step': step, 'model': model.state_dict(), 'code_fp': codebook_fingerprint(model),
                        'optimizer': optimizer.state_dict(),
                        'param_names': [ _pn.get(id(p), 'external') for _pn in [{id(pp): n for n, pp in model.named_parameters()}] for p in [p for g in optimizer.param_groups for p in g['params']] ],
                        'scheduler': scheduler.state_dict(),
                        'best_val_loss': best_val_loss, 'cfg': cfg,
                        'reasoning_enabled_step': reasoning_enabled_step,
                        'recover_count': watchdog.recover_count,
                        'active_depth': depth.active,
                        # M12: full restart state in the ONE checkpoint.
                        'detector': watchdog.state_dict(), 'depth_state': depth.get_state(),
                        'balancer': balancer.state_dict(),
                        'stream_idx': int(stream_idx), 'offset': int(offset),
                        'rng': torch.get_rng_state(), 'data_rng': rng.get_state(),
                        'stream_state': _d15(state), 'stream_gs': _d15(gs if gs is not None else None),
                        'cuda_rng': torch.cuda.get_rng_state() if device == 'cuda' else None,
                    }, _tmp_path)
                    import shutil; shutil.move(_tmp_path, _save_path)
                    print(f'  EVAL saved best.pt (val_loss={val_loss:.4f}) step={step}', flush=True)
                else:
                    print(f'  EVAL no-improve (best={best_val_loss:.4f})', flush=True)
            else:
                print(f'  EVAL step={step}: NO HOLD-OUT DATA (streams empty) - skipping val_loss', flush=True)
                print(f'  NOTE: training appears to be on RANDOM data (no token_stream_*.bin found)', flush=True)
                depth.update(step, None)
            _lcache = getattr(model, 'logit_cache', None)
            if _lcache is not None:
                _lcache.cache.clear()
            model.restore_runtime_buffers(_rt_snap)  # (audit M8 eval isolation)
            model.train()
            gc.collect()
            if device == 'cuda':
                torch.cuda.empty_cache()

        # Checkpoint policy (audit M12, single file): best.pt is the ONLY
        # checkpoint. Val-improving EVAL writes carry the complete restart
        # state (weights, optimizer+param_names, scheduler, watchdog &
        # balancer statistics, data cursor, RNG streams); a step-0 seed
        # written in setup backs the detector before the first val eval.

except KeyboardInterrupt:
    print('Interrupted - saving latest state to best.pt...')
    model.flush_control_pending()   # B16
    _save_path = os.path.join(SAVE_DIR, 'best.pt')
    _tmp_path = _save_path + '.tmp'
    torch.save({
        'step': step, 'model': model.state_dict(), 'code_fp': codebook_fingerprint(model),
        'optimizer': optimizer.state_dict(),
        'param_names': [ _pn.get(id(p), 'external') for _pn in [{id(pp): n for n, pp in model.named_parameters()}] for p in [p for g in optimizer.param_groups for p in g['params']] ],
        'scheduler': scheduler.state_dict(),
        'best_val_loss': best_val_loss, 'cfg': cfg,
        'active_depth': depth.active,
        'recover_count': watchdog.recover_count,   # same full payload as EVAL
        'reasoning_enabled_step': reasoning_enabled_step,
        'detector': watchdog.state_dict(), 'depth_state': depth.get_state(),
        'balancer': balancer.state_dict(),
        'stream_idx': int(stream_idx), 'offset': int(offset),
        'rng': torch.get_rng_state(), 'data_rng': rng.get_state(),
                        'stream_state': _d15(state), 'stream_gs': _d15(gs if gs is not None else None),
                        'cuda_rng': torch.cuda.get_rng_state() if device == 'cuda' else None,
    }, _tmp_path)
    import shutil; shutil.move(_tmp_path, _save_path)
    print(f'Saved latest to best.pt (step {step})')

if _alarm_stop:
    print('STOPPED ON ALARM (D6+, confirmed twice): best.pt = last CLEAN val-save. Inspect the '
          '[ALARM] line above, fix the cause, resume with FORCE_FRESH=False.')
else:
    print('Training complete!')
